# DS Week 06 | Drift-Aware Multi-Station PM2.5 Forecasting
## 1. Business/scientific framing and hypothesis
**Decision:** issue reliable 24-hour-ahead air-pollution forecasts with 90% predictive intervals. An alert system cares about missed high-PM2.5 events, interval sharpness, and stable coverage across stations/seasons—not just average RMSE.

**Hypotheses:** (H1) weather/pollutant lag features beat seasonal persistence; (H2) nonlinear boosting improves over Ridge; (H3) raw quantile bands are undercalibrated under drift; (H4) delayed-feedback online conformal adjustment improves late-test reliability; (H5) pooled coverage hides worse performance at unseen stations and high-pollution extremes.

**Evidence boundary:** this notebook adapts conformal inference ideas from Gibbs & Candès and the 2026 BC-ACI preprint. It does not reproduce published paper experiments or claim formal conditional coverage under nonexchangeable time series.

## 2. Provenance, license, target and data contract
**Primary:** UCI Beijing Multi-Site Air Quality, CC BY 4.0, 12 monitoring sites, 420,768 hourly records, 2013–2017; see `DATASET.md`. No private data or paid APIs. If raw CSVs are absent, deterministic synthetic data is generated solely for an executable smoke run.

At origin `t`, all features are observed at/before `t`; target is PM2.5 at `t+24h`. **Do not substitute actual future weather for a weather forecast.**

In [ ]:
from pathlib import Path
import os,sys,json,time
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error
from sklearn.inspection import permutation_importance
ROOT=Path.cwd().parent if Path.cwd().name=="notebooks" else Path.cwd()
sys.path.insert(0,str(ROOT))
from src.forecasting import *
CFG=json.loads((ROOT/"configs/config.json").read_text());SEED=CFG["seed"]
np.random.seed(SEED);rng=np.random.default_rng(SEED)
FAST=os.environ.get("FAST_MODE","0")=="1"
print("ROOT",ROOT,"FAST_MODE",FAST)

In [ ]:
raw=load_uci(ROOT/"dataset/raw")
if raw is None:
    raw=synthetic_panel(seed=SEED,n_stations=CFG["synthetic_stations"],start=CFG["synthetic_start"],end=CFG["synthetic_end"])
    SOURCE="SYNTHETIC_SMOKE"
else: SOURCE="UCI_BEIJING_12_STATIONS"
print(SOURCE,raw.shape,raw.station.nunique(),raw.ts.min(),raw.ts.max())
assert raw.station.nunique()>=3

## 3. Data-quality EDA: missingness, outliers, site variability and seasonality
NA measurements are preserved; no backward filling or interpolation from the future. Extreme PM2.5 values are retained for risk evaluation unless physically impossible (<0). Plot distributions and monthly patterns; check station availability.

In [ ]:
print("missing fractions",raw[["PM2.5","PM10","TEMP","WSPM","NO2"]].isna().mean().round(3).to_dict())
print("PM2.5 percentiles",raw["PM2.5"].quantile([0,.01,.5,.95,.99,1]).to_dict())
print("station coverage",raw.groupby("station")["PM2.5"].agg(["count","median"]).round(1).to_string())
monthly=raw.assign(month=raw.ts.dt.to_period("M").astype(str)).groupby("month")["PM2.5"].median()
fig,ax=plt.subplots(figsize=(9,3));monthly.plot(ax=ax);ax.set(title=f"Monthly PM2.5 median ({SOURCE})",ylabel="µg/m³");fig.tight_layout();fig.savefig(ROOT/"artifacts/monthly_pm25.png",dpi=110);plt.close(fig)

## 4. Leakage audit and domain-aware features
For each station, create a **regular hourly grid before shift**. Historical PM2.5 lags (1/3/24/168h), 6/24/168h trailing means, 24h volatility/change, current PM2.5-to-PM10 ratio, NO2/O3/CO/SO2, dew-point spread, ventilation proxy, wind vector and cyclical hour/year encode persistence, chemistry, meteorology and seasonal transport. All rolling windows are trailing and include no value after forecast origin. Missing values are imputed with **training-only** medians; unseen station category ignored. Target labels remain missing rather than imputed.

In [ ]:
panel=features(raw,CFG["horizon_hours"])
parts=split_panel(panel,CFG["heldout_stations"],CFG["split_fractions"])
for name in ["train","val","cal","test"]:print(name,len(parts[name]),parts[name].ts.min(),parts[name].ts.max())
print("heldout stations",sorted(parts["heldout"]))
assert parts["train"].label_ts.max()<=parts["cuts"][0]
assert parts["val"].label_ts.max()<=parts["cuts"][1]
assert parts["cal"].label_ts.max()<=parts["cuts"][2]
FEATURES=feature_columns(panel);NUM=[c for c in FEATURES if c!="station"]
print("feature count",len(FEATURES),FEATURES)
assert not {"target","ts","label_ts"}.intersection(FEATURES)

## 5. Model design, baselines and constrained hyperparameter strategy
Seasonal persistence predicts the 24h-ago observed PM2.5 value (at origin `t` the column `pm25_lag_24`). Ridge captures additive signals; HistGradientBoosting handles nonlinear weather/chemistry interactions. Fit on training stations only, select **point model using validation MAE**, never test. Full runs use <=90 boosting iterations; FAST_MODE <=35. A compact hyperparameter sweep can be performed on validation, but test remains untouched.

In [ ]:
pre=ColumnTransformer([("numeric",Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True)),("scale",StandardScaler())]),NUM),
    ("station",OneHotEncoder(handle_unknown="ignore",sparse_output=False),["station"])],sparse_threshold=0)
pre_tree=ColumnTransformer([("numeric",SimpleImputer(strategy="median",add_indicator=True),NUM),
    ("station",OneHotEncoder(handle_unknown="ignore",sparse_output=False),["station"])],sparse_threshold=0)
NITER=CFG["fast_mode_iterations"] if FAST else CFG["boosting_iterations"]
models={"ridge":Pipeline([("prep",pre),("model",Ridge(alpha=40.))]),
 "histgb":Pipeline([("prep",pre_tree),("model",HistGradientBoostingRegressor(max_iter=NITER,max_leaf_nodes=19,learning_rate=.075,l2_regularization=2.,random_state=SEED))])}
train,val,cal,test=[parts[k] for k in ["train","val","cal","test"]]
Xtr=train[FEATURES];ytr=train.target.to_numpy(dtype=np.float32)
Xva=val[FEATURES];yva=val.target.to_numpy(dtype=np.float32)
val_naive=val["pm25_lag_24"].fillna(train["pm25_lag_24"].median()).to_numpy()
val_scores={"seasonal_naive":{"MAE":float(mean_absolute_error(yva,val_naive)),"RMSE":float(np.sqrt(mean_squared_error(yva,val_naive))),"fit_eval_seconds":0.}}
for name,m in models.items():
    t0=time.perf_counter();m.fit(Xtr,ytr);pred=m.predict(Xva)
    val_scores[name]={"MAE":float(mean_absolute_error(yva,pred)),"RMSE":float(np.sqrt(mean_squared_error(yva,pred))),"fit_eval_seconds":round(time.perf_counter()-t0,2)}
print("validation",val_scores)
selected=min(val_scores,key=lambda k:val_scores[k]["MAE"])
model=models.get(selected)
def predict_point(frame):
    if selected=="seasonal_naive":
        return frame["pm25_lag_24"].fillna(train["pm25_lag_24"].median()).to_numpy()
    return model.predict(frame[FEATURES])
print("selected on validation",selected)

## 6. Point forecast evaluation, station/exceedance slices
Compute MAE and RMSE on test. **High-pollution** threshold is the training target 90th percentile, not a threshold tuned on test. Held-out stations have prior telemetry but zero fitting/calibration labels; this is spatial-transfer stress, not sensor-free cold-start.

In [ ]:
pred=predict_point(test);y=test.target.to_numpy()
naive=test["pm25_lag_24"].fillna(train["pm25_lag_24"].median()).to_numpy()
def point_metrics(a,b):return {"MAE":float(mean_absolute_error(a,b)),"RMSE":float(np.sqrt(mean_squared_error(a,b)))}
point_results={"seasonal_naive":point_metrics(y,naive),selected:point_metrics(y,pred)}
print("point test",point_results)
high_threshold=float(train.target.quantile(.90))
for label,mask in [("seen",~test.station.isin(parts["heldout"])),("heldout",test.station.isin(parts["heldout"])),("high_pm25",test.target>=high_threshold)]:
    if mask.any():print(label,int(mask.sum()),point_metrics(y[mask],pred[mask]))

## 7. Train-only feature importance diagnostic
Permutation importance is calculated on a fixed seeded **validation subsample** for computational tractability. It is descriptive and not used to tune features on the test set; correlated lags can dilute individual permutation importance.

In [ ]:
ix=rng.choice(len(val),size=min(1500,len(val)),replace=False)
if model is not None:
    pi=permutation_importance(model,Xva.iloc[ix],yva[ix],n_repeats=2,random_state=SEED,scoring="neg_mean_absolute_error")
    importance=sorted(zip(FEATURES,pi.importances_mean),key=lambda x:-x[1])[:12]
else:
    importance=[("pm25_lag_24",1.0)]
print("validation permutation importance",importance)

## 8. Probabilistic base model: quantile gradient boosting
Fit 5th and 95th percentile regressors on **training only**, then use the independent calibration split to correct their empirical coverage via conformalized quantile regression (CQR). Raw quantiles alone do not guarantee coverage under shift.

In [ ]:
qmodels={}
for quant in [.05,.95]:
    m=Pipeline([("prep",pre_tree),("model",HistGradientBoostingRegressor(loss="quantile",quantile=quant,max_iter=NITER,max_leaf_nodes=19,learning_rate=.075,l2_regularization=2.,random_state=SEED))])
    m.fit(Xtr,ytr);qmodels[quant]=m
lo_cal=qmodels[.05].predict(cal[FEATURES]);hi_cal=qmodels[.95].predict(cal[FEATURES])
lo_test=qmodels[.05].predict(test[FEATURES]);hi_test=qmodels[.95].predict(test[FEATURES])
lo_cal,hi_cal=np.minimum(lo_cal,hi_cal),np.maximum(lo_cal,hi_cal)
lo_test,hi_test=np.minimum(lo_test,hi_test),np.maximum(lo_test,hi_test)
print("base quantile interval",interval_metrics(y,lo_test,hi_test,CFG["alpha"]))

## 9. Calibration: split conformal, CQR and delayed-feedback online CQR
**Split conformal:** calibrate absolute point residuals. **CQR:** calibrate nonconformity `max(lower-y, y-upper)` and add the finite-sample corrected quantile to both sides. **Delayed online CQR:** maintain a bounded rolling score window and adjust alpha based only on forecast errors observed **at least 24 hours after origin**. Because time dependence violates classical exchangeability, evaluate empirical coverage and avoid claiming finite-sample guarantees. No future test labels may enter an interval at forecast time.

In [ ]:
alpha=CFG["alpha"]
cal_pred=predict_point(cal);q_abs=conformal_quantile(np.abs(cal.target.to_numpy()-cal_pred),alpha)
abs_lo=pred-q_abs;abs_hi=pred+q_abs
cal_scores=np.maximum(lo_cal-cal.target.to_numpy(),cal.target.to_numpy()-hi_cal)
q_cqr=conformal_quantile(cal_scores,alpha)
cqr_lo=lo_test-q_cqr;cqr_hi=hi_test+q_cqr
online_lo,online_hi,online_audit=delayed_adaptive_cqr(test.ts,test.label_ts,y,lo_test,hi_test,cal_scores,
    alpha=alpha,eta=CFG["adaptive_eta"],horizon_hours=CFG["horizon_hours"],window=CFG["rolling_calibration_max"])
intervals={"raw_quantiles":(lo_test,hi_test),"split_conformal":(abs_lo,abs_hi),"cqr":(cqr_lo,cqr_hi),"delayed_adaptive_cqr":(online_lo,online_hi)}
interval_results={name:interval_metrics(y,l,h,alpha) for name,(l,h) in intervals.items()}
print("interval results",json.dumps(interval_results,indent=2))
print("online audit",online_audit)
assert online_audit["min_feedback_delay_hours"]>=CFG["horizon_hours"]

## 10. Slice analysis: per-station, unseen station and extreme-event calibration
Marginal coverage can mask dangerously undercovered high-pollution episodes or transferred sites. Report empirical coverage/width for each station and high-pollution subset. Threshold chosen from train to prevent test-target tuning.

In [ ]:
slice_rows=[]
for name,mask in [(f"station:{s}",test.station==s) for s in sorted(test.station.unique())]+[
    ("heldout",test.station.isin(parts["heldout"])),("seen",~test.station.isin(parts["heldout"])),("high_pollution",test.target>=high_threshold)]:
    if not mask.any():continue
    mask=np.asarray(mask)
    for method,(l,h) in intervals.items():slice_rows.append({"slice":name,"method":method,**interval_metrics(y[mask],l[mask],h[mask],alpha)})
slices=pd.DataFrame(slice_rows)
print(slices.loc[slices.method=="delayed_adaptive_cqr"].round(3).to_string(index=False))

## 11. Robustness: seasonal drift, missing sensor data, interval-width trade-off
Assess monthly coverage of adaptive CQR and sensitivity to 10% extra missing observed-weather values at inference. Recompute point predictions using training-fit imputers, without retraining or looking ahead. This stress test measures model reliance on current sensor completeness; it is not a causal missingness model.

In [ ]:
tmp=test.copy();tmp["month"]=tmp.ts.dt.to_period("M").astype(str)
monthly_cov=[]
for month,ix in tmp.groupby("month").indices.items():
    a=np.asarray(ix);monthly_cov.append((month,interval_metrics(y[a],online_lo[a],online_hi[a],alpha)["coverage"]))
print("monthly adaptive coverage",monthly_cov)
stress=test[FEATURES].copy();mask=rng.random(len(stress))<.10
stress.loc[mask,"TEMP"]=np.nan
print("10% TEMP missing stress MAE",point_metrics(y,(model.predict(stress) if model is not None else stress["pm25_lag_24"].fillna(train["pm25_lag_24"].median()).to_numpy())))
fig,ax=plt.subplots(figsize=(9,3));ax.plot([x[0] for x in monthly_cov],[x[1] for x in monthly_cov],marker="o");ax.axhline(1-alpha,ls="--",color="gray");ax.tick_params(axis="x",rotation=45);ax.set(ylabel="empirical coverage",title="Delayed adaptive CQR by test month");fig.tight_layout();fig.savefig(ROOT/"artifacts/monthly_coverage.png",dpi=110);plt.close(fig)

## 12. Uncertainty and statistical reasoning
CQR uses a calibration split disjoint from fitting/selection; **serial dependence and distribution shift invalidate exact classical exchangeability guarantees**. Adaptive feedback is delayed by the forecast horizon. For formal uncertainty on test metrics, use moving-block or station-block bootstrap, not independent row resampling. A quick 7-day moving-block bootstrap of paired MAE differences is included below; overlapping blocks are a descriptive approximation.

In [ ]:
# Weekly time-block bootstrap: paired improvement over seasonal naive, sampled by calendar week.
week=test.ts.dt.to_period("W").astype(str).to_numpy();blocks=[]
for key in np.unique(week):
    ii=np.where(week==key)[0]
    if len(ii):blocks.append(np.mean(np.abs(y[ii]-naive[ii])-np.abs(y[ii]-pred[ii])))
boot=np.array([np.mean(rng.choice(blocks,len(blocks),replace=True)) for _ in range(300)])
print("weekly-block MAE improvement CI",np.quantile(boot,[.025,.975]).tolist())

## 13. Comparison, limitations, research and production follow-ups
Compare point accuracy **and** interval score, coverage, width and slice stability. Pollution alerting requires cost-sensitive thresholds and externally validated health policy. Historical station data may drift; extreme pollution is rare; observed meteorology is not future meteorological forecasts. This is a **single 24-hour horizon** experiment, not a reproduction of BC-ACI's multi-horizon algorithm. Extensions: weather forecast inputs, conformal risk control for exceedance alerts, graph station embeddings, residual bias re-centering, spatiotemporal neural baselines, causal intervention evaluation, prequential monitoring, automated recalibration, and spatially stratified conformal sets.

## 14. Artifact manifest and deterministic experiment report
Persist metrics, data source, split boundaries, features, station holdouts and delay audit. Do not commit raw UCI CSVs or fitted weights. Figures and JSON are small, reproducible outputs.

In [ ]:
report={"project":"DS Week 06","source":SOURCE,"seed":SEED,"horizon_hours":CFG["horizon_hours"],"alpha":alpha,
    "heldout_stations":sorted(parts["heldout"]),"split_cuts":[str(x) for x in parts["cuts"]],
    "rows":{k:len(parts[k]) for k in ["train","val","cal","test"]},"features":FEATURES,
    "validation":val_scores,"selected_point_model":selected,"test_point":point_results,
    "test_intervals":interval_results,"online_feedback_audit":online_audit,
    "high_pm25_threshold_train_p90":high_threshold,"weekly_block_bootstrap_improvement_ci95":np.quantile(boot,[.025,.975]).tolist(),
    "interpretation":"SYNTHETIC scores are only a smoke test" if SOURCE.startswith("SYNTHETIC") else "Real UCI dataset; single dataset study, no causal claims"}
(ROOT/"artifacts/metrics.json").write_text(json.dumps(report,indent=2,default=float))
slices.to_csv(ROOT/"artifacts/slice_metrics.csv",index=False)
print(json.dumps({k:report[k] for k in ["source","selected_point_model","test_point","test_intervals","online_feedback_audit"]},indent=2))